In [1]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, pandas as pd
from memoria_icm.datos import *
from memoria_icm.graficos import *
from memoria_icm.funciones import *
from memoria_icm.funciones_CIR import *
from scipy.optimize import minimize

In [2]:
df, df_resumen = cargar_df_analizados_y_resumen()

Datos cargados. Trackeo: (51917, 27), Resumen: (225, 31)


In [3]:
#para el OU
import numpy as np
import pandas as pd

def estimar_sigma_OU(df, col_x='theta_r', col_t='t', col_id='id'):
    """
    Estima sigma en un modelo Ornstein-Uhlenbeck usando variación cuadrática.
    
    Parámetros:
    - df: DataFrame con columnas de posición (x), tiempo (t) e identificador de trayectoria (id).
    - col_t: nombre de la columna con los tiempos.
    - col_id: nombre de la columna con los identificadores de trayectoria.
    
    Retorna:
    - DataFrame con columnas sigma_hat y sigma2_hat por trayectoria.
    """
    df = df.sort_values(by=[col_id, col_t]).copy()

    def calcular_sigma(grupo):
        x = grupo[col_x].values
        t = grupo[col_t].values
        dx = np.diff(x)
        dt = np.diff(t)

        T_total = np.sum(dt)
        QV = np.sum(dx**2)

        sigma2_hat = QV / T_total if T_total > 0 else np.nan
        sigma_hat = np.sqrt(sigma2_hat) if sigma2_hat >= 0 else np.nan

        grupo['sigma2_hat'] = sigma2_hat
        grupo['sigma_hat'] = sigma_hat
        return grupo

    df = df.groupby(col_id, group_keys=False).apply(calcular_sigma)
    return df


In [5]:
df = estimar_sigma_OU(df)
df

C:\Users\franc\AppData\Local\Temp\ipykernel_21216\766251180.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(col_id, group_keys=False).apply(calcular_sigma)


,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,...,dr,theta_r,v_angular,a_angular,theta_v,curvatura,duracion_video_recortada,nivel_salinidad,sigma2_hat,sigma_hat
0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,0.098320,0.000000,-12.040794,-51.274027,-0.881077,0.000000,11.515,normal,0.168071,0.409965
1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,0.138270,-0.881077,-15.792735,60.562228,-2.033736,126.445970,11.515,normal,0.168071,0.409965
2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,0.088876,-1.812237,-8.807270,142.715592,-2.177735,147.635591,11.515,normal,0.168071,0.409965
3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,0.027576,-1.690928,-0.265367,107.160874,-1.927719,764.513321,11.515,normal,0.168071,0.409965
4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,0.071411,-1.780087,3.049804,56.515627,-0.930846,305.291105,11.515,normal,0.168071,0.409965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51912,320,11.291740,2.965504,-3.120556,3,LA,R,35,Video_006,Larva equinopluteus,...,0.126393,-2.648078,0.006991,0.096506,-2.552953,16.924950,11.515,normal,0.577865,0.760174
51913,320,11.348250,2.959012,-3.125446,3,LA,R,35,Video_006,Larva equinopluteus,...,0.101667,-2.647365,0.021723,0.140003,-2.293583,78.380508,11.515,normal,0.577865,0.760174
51914,320,11.393450,2.957334,-3.128929,3,LA,R,35,Video_006,Larva equinopluteus,...,0.169209,-2.646054,0.023688,-0.067894,-2.407994,75.171379,11.515,normal,0.577865,0.760174
51915,320,11.429330,2.950067,-3.134238,3,LA,R,35,Video_006,Larva equinopluteus,...,0.201462,-2.645356,0.018080,-0.110120,-2.490801,11.765173,11.515,normal,0.577865,0.760174


In [8]:
def estimar_sigma_OU2(df, col_x='theta_r', col_t='t', col_id='id'):
    df = df.sort_values(by=[col_id, col_t]).copy()

    def calcular_sigma(grupo):
        x = grupo[col_x].values
        t = grupo[col_t].values
        dx = np.diff(x)
        dt = np.diff(t)

        numerador = np.sum(dx**2)
        denominador = np.sum(dt)
        sigma2_hat = numerador / denominador if denominador > 0 else np.nan
        sigma_hat = np.sqrt(sigma2_hat) if sigma2_hat >= 0 else np.nan

        grupo['sigma2_hat2'] = sigma2_hat
        grupo['sigma_hat2'] = sigma_hat
        return grupo

    df = df.groupby(col_id, group_keys=False).apply(calcular_sigma)
    return df

df = estimar_sigma_OU2(df)

C:\Users\franc\AppData\Local\Temp\ipykernel_21216\468172063.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(col_id, group_keys=False).apply(calcular_sigma)


In [10]:
df['diferencia1'] = df['sigma2_hat'] - df['sigma2_hat2']
df['diferencia2'] = df['sigma_hat'] - df['sigma_hat2']

In [11]:
df

,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,...,theta_v,curvatura,duracion_video_recortada,nivel_salinidad,sigma2_hat,sigma_hat,sigma2_hat2,sigma_hat2,diferencia1,diferencia2
0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.881077,0.000000,11.515,normal,0.168071,0.409965,0.168071,0.409965,0.0,0.0
1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-2.033736,126.445970,11.515,normal,0.168071,0.409965,0.168071,0.409965,0.0,0.0
2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-2.177735,147.635591,11.515,normal,0.168071,0.409965,0.168071,0.409965,0.0,0.0
3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-1.927719,764.513321,11.515,normal,0.168071,0.409965,0.168071,0.409965,0.0,0.0
4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,Larva recién eclosionada,...,-0.930846,305.291105,11.515,normal,0.168071,0.409965,0.168071,0.409965,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51912,320,11.291740,2.965504,-3.120556,3,LA,R,35,Video_006,Larva equinopluteus,...,-2.552953,16.924950,11.515,normal,0.577865,0.760174,0.577865,0.760174,0.0,0.0
51913,320,11.348250,2.959012,-3.125446,3,LA,R,35,Video_006,Larva equinopluteus,...,-2.293583,78.380508,11.515,normal,0.577865,0.760174,0.577865,0.760174,0.0,0.0
51914,320,11.393450,2.957334,-3.128929,3,LA,R,35,Video_006,Larva equinopluteus,...,-2.407994,75.171379,11.515,normal,0.577865,0.760174,0.577865,0.760174,0.0,0.0
51915,320,11.429330,2.950067,-3.134238,3,LA,R,35,Video_006,Larva equinopluteus,...,-2.490801,11.765173,11.515,normal,0.577865,0.760174,0.577865,0.760174,0.0,0.0
